# Angular and time projections

In this study, we analyze and validate the angle-delay projections by computing the angle and/or delay power profiles in a simulated scenario.

## Setup environment and scenario

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from src.scenarios import get_scenario, ScenarioTheStrandMetadata

CARRIER_FREQUENCY = 6e9  # 6 GHz
N_ELEMENTS_PER_ARRAY_AXIS = 8
MAX_DEPTH_PATH = 5
NUM_SAMPLES_PATH = int(1e3)

# Change default orientation of arrays
ORIENTATION_RX = tf.constant([0, 0, 0], dtype=tf.float32)
ORIENTATION_TX = tf.constant([0, 0, 0], dtype=tf.float32)
# ORIENTATION_RX = tf.constant([0, 0, np.pi/3], dtype=tf.float32)
# ORIENTATION_TX = tf.constant([-np.pi/4, np.pi/2, 0], dtype=tf.float32)

scenario_metadata = ScenarioTheStrandMetadata(
    carrier_frequency=CARRIER_FREQUENCY,
    num_rows_rx_array=N_ELEMENTS_PER_ARRAY_AXIS,
    num_cols_rx_array=N_ELEMENTS_PER_ARRAY_AXIS,
    num_rows_tx_array=N_ELEMENTS_PER_ARRAY_AXIS,
    num_cols_tx_array=N_ELEMENTS_PER_ARRAY_AXIS,
    nb_receivers=1,
    force_rx_positions=[
        [-78.09, -56.81, 1.],
    ]
)

scene = get_scenario(scenario_metadata)

for _, rx in scene.receivers.items():
    rx.orientation = ORIENTATION_RX
for _, tx in scene.transmitters.items():
    tx.orientation = ORIENTATION_TX



## Simualte propagation paths

In [ ]:
%autoreload 2

paths = scene.compute_paths(
    max_depth=MAX_DEPTH_PATH,
    method="fibonacci",
    num_samples=NUM_SAMPLES_PATH,
    diffraction=False, scattering=False, edge_diffraction=False,
    check_scene=False
)
paths.normalize_delays = False

 ## Render setting

In [ ]:
%autoreload 2

from src.scenarios.the_strand.const import TOP_CAM_ZOOM_NAME

fig = scene.render(
    TOP_CAM_ZOOM_NAME,
    show_devices=True,
    paths=paths,
    show_paths=True,
    num_samples=64
)

## Angular projection

We start by selecting a specific path from a given (Rx, Tx) pair.

In [ ]:
IDX_RX = 0
IDX_TX = 0
IDX_PATH = 2

### Plot path in XY and XZ planes

In [ ]:
%autoreload 2

from study.projections.plot_path import plot_path
    

_ = plot_path(
    scene=scene,
    paths=paths,
    idx_path=IDX_PATH,
    idx_rx=IDX_RX,
    idx_tx=IDX_TX
)


### Plot heatmap of received/transmitted power for different steering angles

In [ ]:
%autoreload 2

from study.projections.steered_power_heatmap import compute_steered_power_heatmap

elevation, azimuth, rx_steered_power, tx_steered_power = compute_steered_power_heatmap(
    scene=scene,
    paths=paths,
    n_points=13*26
)

In [ ]:
%autoreload 2

from study.projections.steered_power_heatmap import plot_steered_power_heatmap


plot_steered_power_heatmap(
    paths=paths,
    elevation=elevation,
    azimuth=azimuth,
    rx_steered_power=rx_steered_power,
    tx_steered_power=tx_steered_power,
    idx_path=IDX_PATH,
    idx_rx=IDX_RX,
    idx_tx=IDX_TX
)

### Create lattice of angular projections within received/transmitted paths angle span

In [ ]:
%autoreload 2

%matplotlib notebook

from study.projections.plot_fibonacci_lattice import plot_fibonacci_lattice
    
    
plot_fibonacci_lattice(
    paths=paths,
    n_points_per_lattice=9,
    idx_rx=IDX_RX,
    idx_tx=IDX_TX
)


In [ ]:
%autoreload 2

# Just for reference, here is a plot of the corresponding Fibonacci sphere

from study.projections.plot_fibonacci_lattice import plot_fibonacci_points


plot_fibonacci_points(
    n_points=316,
    min_elevation=tf.constant(0, dtype=tf.float32),
    max_elevation=tf.constant(np.pi, dtype=tf.float32),
    min_azimuth=tf.constant(-np.pi, dtype=tf.float32),
    max_azimuth=tf.constant(np.pi, dtype=tf.float32),
)


 ## Time projection

In [ ]:
IDX_RX = 0
IDX_TX = 0
IDX_PATH = 3

 ### Visualize channel impulse response as a function of the delay

In [ ]:
%autoreload 2

from study.projections.time_projection_plot import plot_cir_vs_delay


plot_cir_vs_delay(
    paths=paths,
    idx_rx=IDX_RX,
    idx_tx=IDX_TX,
#     idx_path=IDX_PATH
)

### Plot projection of CFR

In [ ]:
%autoreload 2

from src.data_classes import MeasurementPhaseNoiseType

from study.projections.time_projection_plot import compute_time_projections, plot_time_projection

paths_projections = False
idx_path=None

num_subcarriers=50
# subcarrier_spacing=30e3  # 30 kHz
subcarrier_spacing=5e6
n_measurements=1
carrier_modulation=True
measurement_snr=100  # 20 dB SNR
component_noise_type=MeasurementPhaseNoiseType.VON_MISES_PHASE
von_mises_prior_concentration=0.0
# component_noise_type=MeasurementPhaseNoiseType.PERFECT_PHASE


projected_measurements = compute_time_projections(
    scene=scene,
    paths=paths,
    num_subcarriers=num_subcarriers,
    subcarrier_spacing=subcarrier_spacing,
    idx_path=idx_path,
    carrier_modulation=carrier_modulation,
    n_measurements=n_measurements,
    component_noise_type=component_noise_type,
    measurement_snr=measurement_snr,
    von_mises_prior_concentration=von_mises_prior_concentration,
    paths_projections=paths_projections,
#     force_n_points=100,
)

plot_time_projection(
    paths=paths,
    projected_measurements=projected_measurements,
    idx_rx=IDX_RX,
    idx_tx=IDX_TX,
    idx_path=idx_path,
    paths_projections=paths_projections
)

## Validate UPEC scheme projections

### Get scene, CFR and metadata from calibration study

In [ ]:
%autoreload 2

from src.scenarios import get_scenario
from src.ofdm_measurements.main import compute_paths_from_metadata, compute_selected_cfr_from_scratch

from src.calibrate_materials.upec.get_projections import get_predicted_paths_projections, get_evenly_spaced_projections
from src.calibrate_materials.upec.compute_projections import compute_projected_power_cfr

from study.calibration.experiment_config import get_measurement_metadata, \
    get_upec_calibration_metadata, get_upec_paths_proj_calibration_metadata


PATHS_PROJECTIONS = True  # If true, project at angles and delays given by the simulated paths


# Metadata
measurement_metadata = get_measurement_metadata(
    measurement_snr=100,  # 20 dB
    measurement_additive_noise=True,
    measurement_perfect_phase=False,
    measurement_von_mises_mean=0.0,
    measurement_von_mises_concentration=0.0,
    bandwidth=6e6  # 6 MHz
)

if PATHS_PROJECTIONS:
    calibration_metadata = get_upec_paths_proj_calibration_metadata(
        normalization_constant=1.0
    )
else:
    calibration_metadata = get_upec_calibration_metadata(
        normalization_constant=1.0
    )

# Scene, paths, and CFR
scene = get_scenario(scenario_metadata)
paths = compute_paths_from_metadata(
    scene=scene,
    measurement_metadata=measurement_metadata
)

cfr_per_mpc = compute_selected_cfr_from_scratch(
    scene=scene,
    measurement_metadata=measurement_metadata,
    normalization_constant=calibration_metadata.normalization_constant
)

# Projections
if PATHS_PROJECTIONS:
    (
        time_projections,  # Shape [N_RX_TX_PAIRS, N_SUBCARRIERS, N_POINTS_TIME]
        rx_angle_projections,  # Shape [N_RX_TX_PAIRS, N_ARR_RX, N_POINTS_ANGLE]
        tx_angle_projections  # Shape [N_RX_TX_PAIRS, N_ARR_TX, N_POINTS_ANGLE]
    ) = get_predicted_paths_projections(
        scene=scene,
        measurement_metadata=measurement_metadata
    )
else:
    (
        time_projections,  # Shape [N_RX_TX_PAIRS, N_SUBCARRIERS, N_POINTS_TIME]
        rx_angle_projections,  # Shape [N_RX_TX_PAIRS, N_ARR_RX, N_POINTS_ANGLE]
        tx_angle_projections  # Shape [N_RX_TX_PAIRS, N_ARR_TX, N_POINTS_ANGLE]
    ) = get_evenly_spaced_projections(
        scene=scene,
        measurement_metadata=measurement_metadata,
        calibration_metadata=calibration_metadata
    )

# Shape [N_MEASUREMENTS=1, N_RX_TX_PAIRS, N_POINTS_TIME, N_POINTS_ANGLE, N_POINTS_ANGLE]
projected_power_cfr = compute_projected_power_cfr(
    cfr_per_mpc=cfr_per_mpc,
    time_projections=time_projections,
    rx_angle_projections=rx_angle_projections,
    tx_angle_projections=tx_angle_projections
)

In [ ]:
%autoreload 2

from study.projections.time_projection_plot import plot_time_projection


IDX_RX_ANGLE_PROJ = None
IDX_TX_ANGLE_PROJ = None

power_vs_time = projected_power_cfr[0, 0]

if IDX_RX_ANGLE_PROJ is None:
    power_vs_time = tf.reduce_mean(power_vs_time, axis=1)
else:
    power_vs_time = power_vs_time[:, IDX_RX_ANGLE_PROJ, :]

if IDX_TX_ANGLE_PROJ is None:
    power_vs_time = tf.reduce_mean(power_vs_time, axis=1)
else:
    power_vs_time = power_vs_time[:, IDX_TX_ANGLE_PROJ]


amplitude_vs_time = tf.sqrt(power_vs_time)

plot_time_projection(
    paths=paths,
    projected_measurements=amplitude_vs_time[tf.newaxis, tf.newaxis, :],
    idx_rx=0,
    idx_tx=0,
    paths_projections=PATHS_PROJECTIONS
)

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from sionna.rt import Paths

from src.utils.telecom_utils import get_lower_bound_angular_resolution_ula
from src.calibrate_materials import CalibrateMaterialsUPECMetadata
from src.projections.angle_lattice import get_paths_angles, get_angle_bounds_paths, get_fibonacci_angle_lattices


def plot_scatter_angle_projection(
    paths: Paths,
    projected_power_cfr: tf.Tensor,  # Shape [N_MEASUREMENTS=1, N_RX_TX_PAIRS, N_POINTS_TIME, N_POINTS_RX_ANGLE, N_POINTS_TX_ANGLE]
    calibration_metadata: CalibrateMaterialsUPECMetadata,
    paths_projections: bool = False,
    time_idx: int = None
):
    if paths_projections:
        rx_aoa_angle_lattices, tx_aod_angle_lattices = get_paths_angles(paths=paths)
    else:
        # Get angle resolution
        angular_resolution = get_lower_bound_angular_resolution_ula(
            n_array_elements=calibration_metadata.n_elements_per_array_axis,
            spacing_array_elements=calibration_metadata.spacing_array_elements
        )

        # Get angle bounds, and number of points in lattice
        _angle_bounds = get_angle_bounds_paths(
            paths=paths,
            angle_resolution_az=angular_resolution,
            angle_resolution_el=angular_resolution
        )

        # Get angle lattice
        rx_aoa_angle_lattices = get_fibonacci_angle_lattices(  # (El, Az) Rx proj ; Shape [N_RX, N_TX, N_POINTS, 2]
            n_points=projected_power_cfr.shape[3],
            min_elevation=_angle_bounds.rx_min_el,
            max_elevation=_angle_bounds.rx_max_el,
            min_azimuth=_angle_bounds.rx_min_az,
            max_azimuth=_angle_bounds.rx_max_az,
        )
        tx_aod_angle_lattices = get_fibonacci_angle_lattices(  # (El, Az) Tx proj ; Shape [N_RX, N_TX, N_POINTS, 2]
            n_points=projected_power_cfr.shape[4],
            min_elevation=_angle_bounds.tx_min_el,
            max_elevation=_angle_bounds.tx_max_el,
            min_azimuth=_angle_bounds.tx_min_az,
            max_azimuth=_angle_bounds.tx_max_az,
        )
    
    # Select first (Rx, Tx) pair
    rx_aoa_angle_lattices, tx_aod_angle_lattices = rx_aoa_angle_lattices[0, 0], tx_aod_angle_lattices[0, 0]
    
    # Get Average power for Tx and Rx individually
    if time_idx is None:  # Average over time
        power_vs_rx_angle = tf.reduce_mean(projected_power_cfr[0, 0], axis=[0, 2])
        power_vs_tx_angle = tf.reduce_mean(projected_power_cfr[0, 0], axis=[0, 1])
    else:  # Select time step
        power_vs_rx_angle = tf.reduce_mean(projected_power_cfr[0, 0, time_idx], axis=[1])
        power_vs_tx_angle = tf.reduce_mean(projected_power_cfr[0, 0, time_idx], axis=[0])
    # To dB
    db_factor = 10 / tf.math.log(tf.constant(10, dtype=tf.float32))
    power_vs_rx_angle = db_factor * tf.math.log(power_vs_rx_angle)
    power_vs_tx_angle = db_factor * tf.math.log(power_vs_tx_angle)
    
    # Scatter plot
    fig, axs = plt.subplots(2, 1)
    fig.set_size_inches((10, 10))
    for ax in axs:
        ax.set_xlabel("Azimuth")
        ax.set_ylabel("Elevation")
    plot_kwargs = dict(
        s=200
    )
    
    # Rx steering
    axs[0].set_title("Power vs receiver steering")
    sc_rx = axs[0].scatter(
        x=tf.math.floormod(rx_aoa_angle_lattices[:, 1], 2*np.pi),
        y=rx_aoa_angle_lattices[:, 0],
        c=power_vs_rx_angle,
        **plot_kwargs
    )
    axs[0].scatter(
        x=tf.math.floormod(paths.phi_r[0, 0, 0, :], 2*np.pi),
        y=paths.theta_r[0, 0, 0, :],
        marker="x",
        color="red",
        **plot_kwargs
    )
    plt.colorbar(sc_rx, ax=axs[0])
    
    # Tx steering
    axs[1].set_title("Power vs transmitter steering")
    sc_tx = axs[1].scatter(
        x=tf.math.floormod(tx_aod_angle_lattices[:, 1], 2*np.pi),
        y=tx_aod_angle_lattices[:, 0],
        c=power_vs_tx_angle,
        **plot_kwargs
    )
    axs[1].scatter(
        x=tf.math.floormod(paths.phi_t[0, 0, 0, :], 2*np.pi),
        y=paths.theta_t[0, 0, 0, :],
        marker="x",
        color="red",
        **plot_kwargs
    )
    plt.colorbar(sc_tx, ax=axs[1])

    
plot_scatter_angle_projection(
    paths=paths,
    projected_power_cfr=projected_power_cfr,
    calibration_metadata=calibration_metadata,
    paths_projections=PATHS_PROJECTIONS,
    time_idx=0
)